# Day 4A — Estimated Lifetime Revenue (LTV Proxy)

**Important limitation:** The Telco dataset does **not** contain true future LTV.
This notebook builds a transparent **LTV Proxy / Estimated Lifetime Revenue** for prioritization — not actual future revenue prediction.

Uses the **saved Day 3 churn model** (no retraining).

In [ ]:
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from churn_model import load_churn_model, load_preprocessor, predict_churn
from ltv_model import (
    LTV_METHODOLOGY,
    add_ltv_columns,
    add_retention_priority,
    assign_ltv_segments,
    high_risk_high_ltv,
)
from preprocessing import create_features, get_feature_matrix

FIGURES = PROJECT_ROOT / "reports" / "figures"
REPORTS = PROJECT_ROOT / "reports"
FIGURES.mkdir(parents=True, exist_ok=True)

print(LTV_METHODOLOGY)

## 1. Load Day 3 model and score all customers

In [ ]:
model = load_churn_model()
preprocessor = load_preprocessor()
meta = joblib.load(PROJECT_ROOT / "models" / "churn_model_metadata.pkl")
print("Model:", type(model).__name__, meta.get("best_model_name"))
print("No retraining.")

cleaned = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "cleaned_telco.csv")
featured = create_features(cleaned)
X, y = get_feature_matrix(featured)
X_proc = pd.DataFrame(preprocessor.transform(X), columns=preprocessor.get_feature_names_out())
preds = predict_churn(model, X_proc)

df = cleaned.copy().reset_index(drop=True)
df["churn_prediction"] = preds["prediction"]
df["churn_probability"] = preds["probability"]
df["risk_level"] = preds["risk_level"]
df.head()

## 2. Calculate Estimated Lifetime Revenue + LTV segments (terciles)

In [ ]:
df = add_ltv_columns(df)
df = assign_ltv_segments(df)
df = add_retention_priority(df)

q33 = df["estimated_ltv"].quantile(1/3)
q66 = df["estimated_ltv"].quantile(2/3)
print(f"Tercile thresholds from data: Low <= {q33:.2f} | Medium <= {q66:.2f} | High > {q66:.2f}")

display(df[["customerID", "tenure", "MonthlyCharges", "estimated_ltv", "ltv_segment", "Churn",
            "churn_probability", "risk_level", "retention_priority"]].head(15))

## 3. LTV visualizations

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df["estimated_ltv"], bins=40, color="steelblue")
plt.title("Estimated Lifetime Revenue (LTV Proxy) Distribution")
plt.tight_layout(); plt.savefig(FIGURES/"ltv_distribution.png", dpi=150, bbox_inches="tight"); plt.show()

plt.figure(figsize=(7,5))
sns.boxplot(data=df, x="Churn", y="estimated_ltv", hue="Churn", palette="Set2", legend=False)
plt.title("Estimated LTV by Churn"); plt.tight_layout()
plt.savefig(FIGURES/"ltv_by_churn.png", dpi=150, bbox_inches="tight"); plt.show()

plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="Contract", y="estimated_ltv", hue="Contract", palette="Set2", legend=False)
plt.title("Estimated LTV by Contract"); plt.tight_layout()
plt.savefig(FIGURES/"ltv_by_contract.png", dpi=150, bbox_inches="tight"); plt.show()

plt.figure(figsize=(7,5))
sns.boxplot(data=df, x="risk_level", y="estimated_ltv", order=["Low","Medium","High"],
            hue="risk_level", hue_order=["Low","Medium","High"], palette="Set2", legend=False)
plt.title("Estimated LTV by Risk Level"); plt.tight_layout()
plt.savefig(FIGURES/"ltv_by_risk_level.png", dpi=150, bbox_inches="tight"); plt.show()

seg = df["ltv_segment"].value_counts().reindex(["Low","Medium","High"])
plt.figure(figsize=(6,4))
sns.barplot(x=seg.index, y=seg.values, hue=seg.index, palette="Blues", legend=False)
plt.title("LTV Segments"); plt.ylabel("Customers"); plt.tight_layout()
plt.savefig(FIGURES/"ltv_segments.png", dpi=150, bbox_inches="tight"); plt.show()

## 4. Retention priority + High-risk / High-LTV customers

Priority rules are documented MVP heuristics (not universal business law).

In [ ]:
retention = df[["customerID","churn_prediction","churn_probability","risk_level",
                "estimated_ltv","ltv_segment","retention_priority"]]
retention.to_csv(REPORTS/"retention_priority.csv", index=False)
print("Priority counts:")
print(df["retention_priority"].value_counts())

hrhl = high_risk_high_ltv(df)
hrhl.to_csv(REPORTS/"high_risk_high_ltv_customers.csv", index=False)
print(f"\nHigh-risk + High-LTV customers: {len(hrhl)}")
display(hrhl.head(20))
print("\nThese customers may deserve higher-priority retention attention.")
print("Retention is not guaranteed to prevent churn.")

df.to_csv(PROJECT_ROOT/"data"/"processed"/"customer_ltv.csv", index=False)
print("Saved customer_ltv.csv and retention_priority.csv")